In [ ]:
import pandas as pd
from langchain_ollama import ChatOllama
from ragas import EvaluationDataset, evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import BleuScore, RougeScore

In [ ]:
# ------------------------------
# 1. Build the Evaluation Dataset
# ------------------------------

# Load the dataset from a CSV file
df = pd.read_csv(
    "../results/Question_Answers_300_0.8_temp_singleline_nomic-embed-text_latest_deepseek-r1_7b.csv"
)

# Build the evaluation dataset using the stored columns.
dataset = []

for idx, row in df.iterrows():
    query = row["Question"]
    print(f"Processing row {idx+1} with query: {query}")

    # Reconstruct the retrieved chunks list.
    # We assume that the stored "Retrieved Chunks" column is a string of chunks separated by "\n\n".
    if isinstance(row["Retrieved Chunks"], str):
        retrieved_contexts = [
            chunk.strip()
            for chunk in row["Retrieved Chunks"].split("\n\n")
            if chunk.strip()
        ]
    else:
        retrieved_contexts = row["Retrieved Chunks"]

    # Build the sample dictionary.
    sample = {
        "user_input": row["Question"],
        "retrieved_contexts": retrieved_contexts,
        "response": row["Generated response"],
        "reference": row["Answer"],
    }
    dataset.append(sample)

# Create an EvaluationDataset from the list of samples.
evaluation_dataset = EvaluationDataset.from_list(dataset)

In [ ]:
# Check the dataset
print(f"Loaded {len(evaluation_dataset)} samples from the dataset.")
print(f"First sample: {evaluation_dataset[0]}")

In [ ]:
# ------------------------------
# 2. Initialize the LLM Wrapper
# ------------------------------

llm = ChatOllama(model="deepseek-r1:7b", temperature=0.3)

# Wrap the LLM with LangchainLLMWrapper for RAGAs.
evaluator_llm = LangchainLLMWrapper(llm)

In [ ]:
# ------------------------------
# Run the Evaluation with the desired metrics.
# ------------------------------
result = evaluate(
    dataset=evaluation_dataset,
    metrics=[BleuScore(), RougeScore(rouge_type="rougeL")],
    llm=evaluator_llm,
)

In [ ]:
print("Evaluation Results:")
print(result)

In [ ]:
result.scores